# Mallory, Hardcastle et al. (2021): the freely moving claims, on the paper's figshare deposits

**Paper.** Mallory CS, Hardcastle K, Campbell MG, Attinger A, Low IIC, Raymond JL, Giocomo LM (2021).
Mouse entorhinal cortex encodes a diverse repertoire of self-motion signals. *Nature Communications*
12, 671. [doi:10.1038/s41467-021-20936-8](https://doi.org/10.1038/s41467-021-20936-8). The claims file
(`claims/mallory-2021.yaml`) was written from the version of record with its Methods, legends and
Supplementary Information.

**Data.** Dandiset 000053 holds no head pitch or roll, so a reader marked the freely moving claims
`data: elsewhere` (question of 2026-11-10), on the paper's own deposits: figshare Dataset 1
([doi:10.6084/m9.figshare.13425392.v1](https://doi.org/10.6084/m9.figshare.13425392.v1),
Freely_moving_data_with_inertial_sensor.mat, Cohort 1, 179 cells with pitch and roll) and Dataset 2
([doi:10.6084/m9.figshare.13425017.v1](https://doi.org/10.6084/m9.figshare.13425017.v1),
Freely_moving_data_without_inertial_sensor.mat, Cohort 2). Both are version 1, the only version.
Dataset 1 is one compressed MATLAB 5 variable, so it is streamed whole into memory (1.7 GB) and
reduced as it loads; Dataset 2 (6.4 GB, MATLAB 7.3) is read in place, only its azimuthal head
velocity, for Figure 1b. Nothing is written to disk. Loading is in `mec_free.py`, which also holds the open field LN model
(run here on Cohort 1).

| Claims | Status |
|---|---|
| fig1b-head-posture-distributions | tested here (Datasets 1 and 2) |
| the claims that need the LN model on Cohort 1 (fig1c-head-direction-cells, fig1c-pitch-roll-cells, fig1c-pitch-roll-velocity-rare, fig1d-conjunctive-head, fig2-pitch-roll-shapes, fig2a, fig2b, fig2-pitch-roll-stability, and the Cohort 1 parts of supp4-head and supp5-head) | tested here, under readings (1a, 2a) only |
| the claims that need the LN model on Cohort 2 (fig1c-azimuthal-velocity-cells and the fig2c and fig3 claims on it, supp9-head) | blocked: too large (one reading of 1021 cells, about 3 h by the Cohort 1 time) |
| supp3-head-low-false-positives | blocked: six more runs of the Cohort 1 LN model, about 3.2 h |
| head-fixed claims | `reproduce_mallory_2021.ipynb` |

In [ ]:
import math
import os
import time
import resource
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats

import mec_ln as L
import mec_free as F

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
STARTED = time.time()
SUBSET = int(os.environ.get("MALLORY_SUBSET", "0"))   # 0: every session (develop on a few)


def gb():
    return round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2)

## Data check

In [ ]:
t0 = time.time()
raw = F.fetch()
print(f"Dataset 1: {len(raw) / 1e9:.3f} GB streamed in {time.time() - t0:.0f} s")
SESS = F.load_dataset1(raw)
del raw
if SUBSET:
    SESS = dict(list(SESS.items())[:SUBSET])
print(f"read in {time.time() - t0:.0f} s; peak memory {gb()} GB")
rows = []
for sid, s in SESS.items():
    b = s["beh"]
    dt = np.diff(s["t"])
    rows.append({"session": sid, "mouse": s["animal"], "arena_cm": s["arena"], "cells": len(s["cells"]),
                 "minutes": round(s["t"][-1] / 60, 1), "dt_min": round(dt.min(), 4), "dt_max": round(dt.max(), 4),
                 "same_behaviour": s["same_behaviour"],
                 "x_range": (round(np.nanmin(b["x"]), 1), round(np.nanmax(b["x"]), 1)),
                 "y_range": (round(np.nanmin(b["y"]), 1), round(np.nanmax(b["y"]), 1)),
                 "nan%": round(100 * max(np.isnan(b[k]).mean() for k in ("x", "y", "Bs", "Ha", "Hp", "Hr")), 2)})
S1 = pd.DataFrame(rows)
print(S1.to_string(index=False))
print("\nCells:", int(S1.cells.sum()), "in", len(S1), "sessions of", S1.mouse.nunique(), "mice;",
      S1.groupby("mouse").cells.sum().to_dict())
print("Every cell of a session carries the same behaviour:", bool(S1.same_behaviour.all()))
print("Arena sides (cm):", dict(Counter(S1.arena_cm)))
allb = {k: np.concatenate([s["beh"][k] for s in SESS.values()]) for k in ("x", "y", "Bs", "Ha", "Had", "Hp", "Hr", "Hpd", "Hrd")}
print("\nRanges over all sessions (1st, 50th, 99th percentile; min, max):")
units = {"x": "cm", "y": "cm", "Bs": "cm/s", "Ha": "rad", "Had": "deg/s", "Hp": "deg", "Hr": "deg", "Hpd": "deg/s", "Hrd": "deg/s"}
for k, v in allb.items():
    print(f"  {k:4s} [{units[k]}]: {np.round(np.nanpercentile(v, [1, 50, 99]), 2)}; {np.nanmin(v):.2f} to {np.nanmax(v):.2f}")
n_sp = np.concatenate([s["n_spikes"] for s in SESS.values()])
n_in = np.concatenate([s["Y"].sum(1) for s in SESS.values()])
print("Spikes outside the behaviour samples:", int((n_sp - n_in).sum()), "of", int(n_sp.sum()))

The deposit's description gives positions in cm, speed in cm/s, head direction in degrees, azimuthal
head velocity in deg/s and pitch and roll in radians. The values agree except for head direction,
which is in radians, not degrees: its 1st to 99th percentiles are 0.05 to 6.23 (a few samples lie
outside 0 to 2 pi, -1.67 to 8.42). Pitch and roll, converted to degrees, have 1st to 99th
percentiles of -41.8 to 68.1 and -69.0 to 75.7 deg. Behaviour is sampled every 20 ms (the Methods'
50 Hz) in every session. The arenas are 50 and 70 cm square, two of the sizes the Methods list. No
spike falls outside the behaviour samples.

In [ ]:
# How the deposit's azimuthal head velocity relates to its head direction, and so how to take the
# pitch and roll velocities the deposit lacks.
s = next(iter(SESS.values()))
ha = np.degrees(s["beh"]["Ha"])
fwd = np.diff(ha) / F.DT
print("Azimuthal head velocity equal to the forward difference of head direction (deg / 20 ms), "
      "where the angle does not wrap:")
nowrap = np.abs(np.diff(ha)) < 180
print("  fraction of samples:", round(float(np.mean(np.isclose(s["beh"]["Had"][:-1][nowrap], fwd[nowrap], atol=1e-6))), 4))
wrap = ~nowrap
print("  where it wraps, |velocity| > 9000 deg/s in", round(float(np.mean(np.abs(s["beh"]["Had"][:-1][wrap]) > 9000)), 3),
      "of", int(wrap.sum()), "samples (the wrap is not undone; these lie outside the inner 95%)")
# Sessions that share their behaviour
keys = list(SESS)
dups = []
for i, a in enumerate(keys):
    for b in keys[i + 1:]:
        A, B = SESS[a]["beh"], SESS[b]["beh"]
        if len(A["x"]) == len(B["x"]) and np.array_equal(A["x"], B["x"], equal_nan=True):
            dups.append((a, b, {k: bool(np.array_equal(A[k], B[k], equal_nan=True)) for k in ("y", "Bs", "Ha", "Hp", "Hr")}))
print("\nSessions with identical body position:", dups)

The deposit's azimuthal head velocity is the forward difference of its head direction, not unwrapped,
so the velocities of pitch and roll are taken the same way (`mec_free`). Two sessions of different
mice (Fuschia 052001 and Daisy 052001) share their body position, speed and head direction exactly,
with different pitch and roll; this is reported as a finding, and both are kept, as the deposit keeps
them.

In [ ]:
# Clock check: spikes, LED tracking and the IMU share a clock if the tuning to body position and to
# pitch is strongest with no shift. Fraction of the variance of each cell's spike count (smoothed
# over 0.5 s) explained by 10 x 10 position bins, or 20 pitch bins, with the behaviour shifted.
LAGS = np.arange(-100, 101, 10)   # 20 ms bins: -2 to +2 s


def eta2(lab, y, nlab):
    n = np.bincount(lab, minlength=nlab)
    s = np.bincount(lab, weights=y, minlength=nlab)
    ok = n > 0
    tot = ((y - y.mean()) ** 2).sum()
    return float(((s[ok] ** 2 / n[ok]).sum() - y.sum() ** 2 / len(y)) / tot) if tot > 0 else np.nan


best = {"B": [], "Hp": []}
for sid, s in SESS.items():
    b = s["beh"]
    ok = np.isfinite(b["x"]) & np.isfinite(b["y"]) & np.isfinite(b["Hp"])
    ix = np.clip(((b["x"] - np.nanmin(b["x"])) / np.ptp(b["x"][ok]) * 10).astype(int), 0, 9)
    iy = np.clip(((b["y"] - np.nanmin(b["y"])) / np.ptp(b["y"][ok]) * 10).astype(int), 0, 9)
    lp = np.percentile(b["Hp"][ok], [1, 99])
    ip = np.clip(((b["Hp"] - lp[0]) / (lp[1] - lp[0]) * 20).astype(int), 0, 19)
    for yc in s["Y"]:
        ys = np.convolve(yc.astype(float), np.ones(25) / 25, "same")
        for var, lab, nl in (("B", ix * 10 + iy, 100), ("Hp", ip, 20)):
            e = []
            for lag in LAGS:
                sl = slice(max(0, lag), len(ys) + min(0, lag))
                sb = slice(max(0, -lag), len(ys) - max(0, lag))
                m = ok[sb]
                e.append(eta2(lab[sb][m], ys[sl][m], nl))
            best[var].append((sid, LAGS[int(np.nanargmax(e))], max(e)))
for var, rows_ in best.items():
    lag = np.array([r[1] for r in rows_]) * F.DT
    top = np.array([r[2] for r in rows_])
    strong = top > np.percentile(top, 75)
    print(f"{var}: lag of best tuning over {len(lag)} cells, median {np.median(lag):+.2f} s; "
          f"cells with the strongest quarter of tuning: median {np.median(lag[strong]):+.2f} s, "
          f"{np.mean(np.abs(lag[strong]) <= 0.2):.2f} within 0.2 s")
    by = pd.DataFrame(rows_, columns=["session", "lag", "eta2"])
    by["mouse"] = by.session.map(lambda x: SESS[x]["animal"])
    print("   median lag by mouse (s):", (by.groupby("mouse").lag.median() * F.DT).round(2).to_dict())

The check does not establish a shared clock. Over the 179 cells the shift that gives the strongest
tuning has a median of -0.20 s for position and -0.40 s for pitch, and among the quarter of cells
with the strongest tuning only 0.18 (position) and 0.24 (pitch) peak within 0.2 s of no shift; the
median shift differs by mouse (-0.4 to +1.4 s for position). With 20 ms bins and a 0.5 s smoothing
of the spike counts this measure is coarse, and most cells are weakly tuned, so it neither shows nor
rules out an offset. fig1b uses the behaviour alone, so it does not depend on the clock. Every
claim on the Cohort 1 LN model (cells 14 to 44) does: a misalignment of spikes and behaviour would
lower the counts of encoding cells and the stability of their tuning. Those outcomes rest on the
deposit's alignment, an assumption stated with each of them.

In [ ]:
# Dataset 2: cells, sessions and mice, read in place.
t0 = time.time()
IDX2 = pd.DataFrame(F.dataset2_index())
print(f"Dataset 2 index read in {time.time() - t0:.0f} s")
print("Cells:", len(IDX2), "; sessions:", IDX2.session.nunique(), "; mice:", IDX2.animal.nunique())
print("Arena sides (cm), by cell:", IDX2.arena.value_counts().to_dict())
print("Mice in both datasets:", sorted(set(IDX2.animal) & set(S1.mouse)))
first = IDX2.groupby("session").row.first()
if SUBSET:
    first = first.iloc[:SUBSET * 2]
t0 = time.time()
HAV2 = F.dataset2_series(first.tolist(), "azimuthal_head_velocity")
print(f"azimuthal head velocity of {len(HAV2)} sessions read in {time.time() - t0:.0f} s; "
      f"1st and 99th percentile over all: {np.round(np.nanpercentile(np.concatenate(list(HAV2.values())), [1, 99]), 1)} deg/s")

### fig1b-head-posture-distributions

> Across Cohort 1 mice the median head pitch is 11.5 deg (inner 95% from -23.0 to 55.6 deg) and the
> median roll -0.8 deg (-50.9 to 43.6 deg); azimuthal head velocity has a median near 0 deg/s and an
> inner 95% from -389.9 to 385.1 deg/s across 25 mice of both cohorts.

Percentiles per session, averaged over each mouse's sessions, then the mean over mice, against the
legend's sem across mice.

In [ ]:
rows = []
for sid, s in SESS.items():
    for v in ("Hp", "Hr", "Had"):
        p = np.nanpercentile(s["beh"][v], [2.5, 50, 97.5])
        rows.append({"mouse": s["animal"], "session": sid, "var": v, "p2.5": p[0], "p50": p[1], "p97.5": p[2], "cohort": 1})
an2 = IDX2.groupby("session").animal.first()
for sid, row in first.items():
    p = np.nanpercentile(HAV2[row], [2.5, 50, 97.5])
    rows.append({"mouse": an2[sid], "session": sid, "var": "Had", "p2.5": p[0], "p50": p[1], "p97.5": p[2], "cohort": 2})
pct = pd.DataFrame(rows)
per_mouse = pct.groupby(["var", "mouse"])[["p2.5", "p50", "p97.5"]].mean()
EXPECTED_1B = [("Hp", "p50", 11.5, 4.1), ("Hp", "p2.5", -23.0, 4.5), ("Hp", "p97.5", 55.6, 4.1),
               ("Hr", "p50", -0.8, 7.6), ("Hr", "p2.5", -50.9, 3.6), ("Hr", "p97.5", 43.6, 4.0),
               ("Had", "p2.5", -389.9, 34.7), ("Had", "p97.5", 385.1, 34.7)]
out = []
for v, q, val, tol in EXPECTED_1B:
    x = per_mouse.loc[v][q]
    out.append({"var": v, "percentile": q, "mean over mice": round(x.mean(), 2), "sem": round(x.sem(), 2),
                "mice": len(x), "paper": val, "tol": tol, "within": abs(x.mean() - val) <= tol})
t1b = pd.DataFrame(out)
print(t1b.to_string(index=False))
print("Within tolerance:", int(t1b.within.sum()), "of 8")
print("Azimuthal head velocity median, mean over mice:", round(per_mouse.loc["Had"]["p50"].mean(), 2), "deg/s")
print("\nPer mouse (deg, deg/s):")
print(per_mouse.round(1).to_string())
print("Sessions per mouse:", pct[pct["var"] == "Had"].groupby("mouse").session.size().to_dict())

**Result: reproduced.** All eight values are within the legend's sem: pitch median 11.45, 2.5th
percentile -23.03 and 97.5th percentile 55.57 deg over 6 mice; roll -0.76, -50.92 and 43.63 deg;
azimuthal head velocity 2.5th and 97.5th percentiles -389.82 and 385.01 deg/s over the 25 mice of
both cohorts, with a median of -0.1 deg/s.

## The open field LN model on Cohort 1, under readings (1a, 2a)

The claims file's LN pipeline (opening comment) over the eight Cohort 1 variables (B, Bs, Ha, Hp, Hr
and the azimuthal, pitch and roll head velocities), for every cell of figshare Dataset 1, with open
detail (1) under reading (a), 10-s sections assigned to folds in turn (compute_test_train_ind.m),
and open detail (2) under reading (a), bins outside any variable's inner 95% dropped. The criteria ask
for both readings of (1) and (2); only this one was run, because one reading of the 179 cells takes
about 31 minutes on the runner's four cores (printed below; reading (2b) keeps all valid bins, about 1.44 times as many, so about 45 minutes) (the cells average 178,000 bins and the 2D position
spline has up to 121 parameters), and the four readings would not fit within the 60-minute notebook
limit or the run's time. The readings not run are named in each claim's notes.

In [ ]:
t0 = time.time()
WORK = {}
for si, (sid, s) in enumerate(SESS.items()):
    prep = F.prepare(s, L.RANGE_DROP)
    folds = L.fold_labels(prep["n_keep"], L.FOLDS_TURN)
    P = L.by_fold(prep, folds)
    P["seed"] = 5000 + si
    WORK[sid] = (P, s["Y"])
RES1 = F.analyse_all(WORK, processes=4)
print(f"LN model, Cohort 1: {sum(len(r) for r in RES1.values())} cells in {(time.time() - t0) / 60:.1f} min; "
      f"peak memory {gb()} GB")
rows = []
for sid, res in RES1.items():
    P = WORK[sid][0]
    for i, r in enumerate(res):
        rows.append({"session": sid, "mouse": SESS[sid]["animal"], "cell": SESS[sid]["cells"][i],
                     "n_spikes": r["n_spikes"], "kept_bins": P["n_keep"], "valid_bins": P["n_valid"],
                     "selected": ",".join(r["selected"]), "encoded": ",".join(r["encoded"]),
                     "encoded_code": ",".join(r["encoded_code"]), **{v: v in r["encoded"] for v in F.VARS}})
C1 = pd.DataFrame(rows)
print("Kept bins per session (fraction of valid): median", round(float((C1.kept_bins / C1.valid_bins).median()), 3),
      "range", round(float((C1.kept_bins / C1.valid_bins).min()), 3), "to", round(float((C1.kept_bins / C1.valid_bins).max()), 3))
print("Cells with no spike in the kept bins:", int((C1.n_spikes == 0).sum()),
      "; selected model failed the final test against 0:", int(((C1.selected != "") & (C1.encoded == "")).sum()))
print("Cells encoding any variable:", int((C1.encoded != "").sum()), "of", len(C1))
print("Counts per variable:", {v: int(C1[v].sum()) for v in F.VARS})
print("Variant, the code's final test on the last model tried:",
      {v: int(C1.encoded_code.str.split(",").apply(lambda x: v in x).sum()) for v in F.VARS})
print("By mouse:")
print(C1.groupby("mouse")[F.VARS].sum().assign(cells=C1.groupby("mouse").size()).to_string())

### fig1c-head-direction-cells

> In Cohort 1, 63 of 179 MEC cells (35%) encode azimuthal head direction.

In [ ]:
n_all = len(C1)
k = int(C1.Ha.sum())
print(f"Ha cells: {k} of {n_all} ({k / n_all:.3f}); paper 63 (tolerance 6.3), 0.35 (tolerance 0.035); "
      f"count within: {abs(k - 63) <= 6.3}, fraction within: {abs(k / n_all - 0.35) <= 0.035}")
print("By mouse (Ha cells, cells):", {m: (int(g.Ha.sum()), len(g)) for m, g in C1.groupby("mouse")})

**Result: reproduced, under readings (1a, 2a).** 58 of 179 cells (0.324) encode Ha, within the tolerance of 63 and of 0.35. The other readings of open details (1) and (2) were not run, for time.

### fig1c-pitch-roll-cells

> In Cohort 1, 46 MEC cells (26%) encode head pitch and 53 (30%) encode head roll.

In [ ]:
kp, kr = int(C1.Hp.sum()), int(C1.Hr.sum())
print(f"Hp cells: {kp} ({kp / n_all:.3f}), paper 46, tolerance 4.6, within {abs(kp - 46) <= 4.6}")
print(f"Hr cells: {kr} ({kr / n_all:.3f}), paper 53, tolerance 5.3, within {abs(kr - 53) <= 5.3}")
print("By mouse (Hp, Hr, cells):", {m: (int(g.Hp.sum()), int(g.Hr.sum()), len(g)) for m, g in C1.groupby("mouse")})

**Result: not reproduced, under readings (1a, 2a).** 37 cells encode pitch (paper 46, tolerance 4.6) and 43 roll (paper 53, tolerance 5.3); both counts are below tolerance. The other readings were not run.

### fig1c-pitch-roll-velocity-rare

> Few Cohort 1 cells encode angular velocity about the pitch (4 of 179) or roll (5 of 179) axes.

Criterion: each count at most twice the paper's (8 for pitch velocity, 10 for roll velocity).

In [ ]:
kpd, krd = int(C1.Hpd.sum()), int(C1.Hrd.sum())
print(f"Pitch velocity cells: {kpd} of {n_all} (paper 4, limit 8): {kpd <= 8}")
print(f"Roll velocity cells: {krd} of {n_all} (paper 5, limit 10): {krd <= 10}")

**Result: reproduced.** 3 cells encode pitch velocity and 5 roll velocity, within the limits of 8 and 10.

### fig1d-conjunctive-head

> Individual Cohort 1 cells commonly encode more than one variable (95 of 179); few encode body
> speed, head direction, pitch, roll or azimuthal head velocity alone (2%, 5%, 1%, 5% and 0%), while
> 15% encode body position alone.

In [ ]:
n_enc = C1.encoded.apply(lambda x: len(x.split(",")) if x else 0)
k_multi = int((n_enc > 1).sum())
alone = {v: round(float((C1.encoded == v).mean()), 3) for v in ("B", "Bs", "Ha", "Hp", "Hr", "Had")}
print(f"Cells encoding more than one variable: {k_multi} (paper 95, tolerance 9.5): {abs(k_multi - 95) <= 9.5}")
print(f"Fraction encoding B alone: {alone['B']} (paper 0.15, tolerance 0.015): {abs(alone['B'] - 0.15) <= 0.015}")
print("Fractions encoding one variable alone:", alone, "(paper Bs 0.02, Ha 0.05, Hp 0.01, Hr 0.05, Had 0.00)")
print("Each of Bs, Ha, Hp, Hr, Had alone at most 5%:", all(alone[v] <= 0.05 for v in ("Bs", "Ha", "Hp", "Hr", "Had")))
print("Most common encoded sets:")
print(C1.encoded.replace("", "(none)").value_counts().head(12).to_string())

**Result: partially reproduced.** 87 cells encode more than one variable (paper 95, within tolerance), but 0.179 encode body position alone (paper 0.15, tolerance 0.015). Each of Bs, Ha, Hp, Hr and azimuthal head velocity alone is at most 5% (0.022, 0.045, 0.011, 0.022, 0.006).

In [ ]:
# Shape measures of every encoded head variable's curve (pitch and roll smoothed first)
frows, FCURVES = [], {}
for sid, res in RES1.items():
    P = WORK[sid][0]
    for i, r in enumerate(res):
        for v in ("Ha", "Hp", "Hr", "Had"):
            if v in r["encoded"]:
                row = {"session": sid, "mouse": SESS[sid]["animal"], "cell": i, "var": v,
                       "stability": r.get("stability", {}).get(v, np.nan)}
                if v != "Ha":
                    f = F.curve_features(r, v, P["lims"], P["vals"][v])
                    FCURVES[(sid, i, v)] = (f.pop("x"), f.pop("curve"))
                    row.update(f)
                frows.append(row)
HF = pd.DataFrame(frows)
for col in ("order", "explored_a", "explored_b", "pref_a", "pref_b", "pref_r"):
    if col not in HF:
        HF[col] = np.nan
print("Encoded head variable instances:", HF["var"].value_counts().to_dict())
print("Explored fraction of the preset range (readings a, b), quartiles:")
print(HF[HF["var"].isin(["Hp", "Hr", "Had"])].groupby("var")[["explored_a", "explored_b"]].quantile([0, 0.25, 0.5, 1]).round(2).unstack().to_string())


def well_fit(df, reading):
    col = "explored_a" if reading == "a" else "explored_b"
    return df[(df[col] >= F.EXPLORED_MIN) & (df.order > 0)]


def binom(k, n):
    return stats.binomtest(int(k), int(n), 0.5).pvalue if n else np.nan


def zbin(k, n):
    return (k - n / 2) / math.sqrt(n / 4) if n else np.nan

### fig2-pitch-roll-shapes

> Nearly all pitch (44 of 46) and roll (52 of 53) tuning curves are fit by a polynomial of order 5
> or lower; most pitch cells are linear (17) or quadratic (16), and most roll cells are quadratic
> (28, with 10 linear).

In [ ]:
EXP2 = [("pitch cells well fit", 44, 4.4), ("roll cells well fit", 52, 5.2), ("linear pitch", 17, 1.7),
        ("quadratic pitch", 16, 1.6), ("linear roll", 10, 1.0), ("quadratic roll", 28, 2.8)]
for rd in ("a", "b"):
    hp, hr = well_fit(HF[HF["var"] == "Hp"], rd), well_fit(HF[HF["var"] == "Hr"], rd)
    n_hp, n_hr = int((HF["var"] == "Hp").sum()), int((HF["var"] == "Hr").sum())
    vals = [len(hp), len(hr), int((hp.order == 1).sum()), int((hp.order == 2).sum()), int((hr.order == 1).sum()),
            int((hr.order == 2).sum())]
    within = [abs(v - e) <= t for v, (_, e, t) in zip(vals, EXP2)]
    maj_p = (vals[2] + vals[3]) > len(hp) / 2
    maj_r = vals[5] > len(hr) / 2
    print(f"Explored reading ({rd}): of {n_hp} pitch and {n_hr} roll cells")
    for (name, e, t), v, w in zip(EXP2, vals, within):
        print(f"   {name}: {v} (paper {e}, tolerance {t}) {'within' if w else 'outside'}")
    print(f"   linear + quadratic the majority of well-fit pitch cells: {maj_p}; quadratic more than half of "
          f"well-fit roll cells: {maj_r}; counts within tolerance: {sum(within)} of 6")
    print("   orders, pitch:", dict(sorted(Counter(hp.order).items())), "; roll:", dict(sorted(Counter(hr.order).items())))
print("Not explored (reading a, b):", {v: (int((g.explored_a < F.EXPLORED_MIN).sum()), int((g.explored_b < F.EXPLORED_MIN).sum()))
                                         for v, g in HF[HF["var"].isin(["Hp", "Hr"])].groupby("var")},
      "; explored but not fit by order 5:", {v: int(((g.explored_a >= F.EXPLORED_MIN) & (g.order == 0)).sum())
                                              for v, g in HF[HF["var"].isin(["Hp", "Hr"])].groupby("var")})

**Result: not reproduced.** 33 or 34 pitch and 41 roll curves are classified; linear plus quadratic is the majority of pitch cells (12 and 13 or 14), but quadratic roll cells (16) are not more than half of 41, and none of the six counts is within tolerance.

### fig2a-pitch-up-down-similar and fig2b-roll-left-right-similar

> Similar proportions of pitch cells prefer upward and downward head tilts (26 of 44 upward).
> Similar proportions of roll cells prefer leftward (contraversive) and rightward (ipsiversive)
> tilts (30 of 52 leftward).

Preference under open detail (5): (a) the sign of the slope for linear curves, the side of the
maximum otherwise; (b) the side with the larger absolute slope of the lines fitted either side of 0;
(r), a variant, the side with the steeper rise away from 0. The deposit does not say which sign of
pitch is upward or of roll leftward; the binomial test does not depend on it, and the count of the
positive side is printed.

In [ ]:
def pref_table(var):
    out = []
    for rd2 in ("a", "b"):
        c = well_fit(HF[HF["var"] == var], rd2)
        for rd in ("a", "b", "r"):
            k = int((c[f"pref_{rd}"] > 0).sum())
            out.append({"explored": rd2, "preference": rd, "n": len(c), "positive side": k,
                        "fraction": round(k / max(len(c), 1), 3), "Z": round(zbin(k, len(c)), 2),
                        "P": float(f"{binom(k, len(c)):.3g}"), "no preference (P >= 0.05)": binom(k, len(c)) >= 0.05})
    return pd.DataFrame(out)


print("Pitch (paper: 26 of 44 upward, Z = 1.06, P = 0.29):")
print(pref_table("Hp").to_string(index=False))

**Result: reproduced.** No reading of preference or explored range finds a pitch preference: 17 to 21 of 33 or 34 cells prefer positive pitch, P from 0.229 to 1.0.

In [ ]:
print("Roll (paper: 30 of 52 leftward, Z = 0.82, P = 0.41):")
print(pref_table("Hr").to_string(index=False))

**Result: reproduced.** No reading finds a roll preference: 21 to 27 of 41 cells prefer positive roll, P from 0.0596 to 1.0.

### fig2-pitch-roll-stability

> Tuning to pitch (median 0.67) and roll (0.54) is about as stable across session halves as tuning to
> azimuthal head direction (0.65).

Stability as in the head-fixed notebook (`mec_free.analyse_cell`): the selected model refitted on each
half of the kept bins, the Pearson correlation of the two model-derived curves on the visualizing
range both halves visited.

In [ ]:
def med(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return (round(float(np.median(x)), 3), round(float(np.percentile(x, 25)), 3), round(float(np.percentile(x, 75)), 3), len(x)) if len(x) else (np.nan, np.nan, np.nan, 0)


ha_st = HF[HF["var"] == "Ha"].stability.values
for rd in ("a", "b"):
    hp_st = well_fit(HF[HF["var"] == "Hp"], rd).stability.values
    hr_st = well_fit(HF[HF["var"] == "Hr"], rd).stability.values
    g = [x[np.isfinite(x)] for x in (hp_st, hr_st, ha_st)]
    kw = stats.kruskal(*g)
    print(f"Explored reading ({rd}): median [IQR, n] pitch {med(hp_st)} (paper 0.67), roll {med(hr_st)} (paper 0.54), "
          f"Ha {med(ha_st)} (paper 0.65)")
    print(f"   Kruskal-Wallis chi2 = {kw.statistic:.2f}, P = {kw.pvalue:.3g} (paper 4.26, 0.12); P >= 0.05: {kw.pvalue >= 0.05}; "
          f"medians within tolerance: pitch {abs(med(hp_st)[0] - 0.67) <= 0.067}, roll {abs(med(hr_st)[0] - 0.54) <= 0.054}, "
          f"Ha {abs(med(ha_st)[0] - 0.65) <= 0.065}")
print("Stability not computable:", int(HF.stability.isna().sum()), "; range:",
      {v: (round(np.nanmin(g.stability), 2), round(np.nanmax(g.stability), 2)) for v, g in HF.groupby("var")})
print("Median by mouse:", HF.groupby(["var", "mouse"]).stability.median().unstack().round(2).to_string())

**Result: reproduced.** Pitch (0.832 or 0.808), roll (0.570) and head direction (0.674) stability do not differ, Kruskal-Wallis P = 0.116 or 0.105. The roll and head direction medians are within tolerance, the pitch median is above it.

### supp4-head-ln-conservative, Cohort 1 part

> For pitch, roll and azimuthal head velocity in the freely moving data, shuffle-based significance
> tests (within-cell and pooled) identify far more cells than the LN model, and nearly all LN model
> cells are also found by shuffling.

As in the head-fixed notebook: the Pearson correlation of each cell's spike count with the variable
over the kept bins, against 500 circular shifts of more than 20 s (2.5th and 97.5th percentiles,
within the cell or pooled over cells); for azimuthal head velocity a cell is also found when the
correlations below and above 0 lie beyond the 5th and 95th percentiles in opposite directions (the
Methods' symmetric test). Cohort 1 only: the claim pools Cohort 2 for azimuthal head velocity, whose
LN model was not run (see the table).

In [ ]:
rng_s4 = np.random.default_rng(42)
S4 = {v: [] for v in ("Hp", "Hr", "Had")}
for si, (sid, res) in enumerate(RES1.items()):
    P = WORK[sid][0]
    inv = np.argsort(P["order"])
    for i, r in enumerate(res):
        y = SESS[sid]["Y"][i, P["keep"]].astype(float)
        for v in S4:
            x = P["vals"][v][inv]
            rr, sh = L.shuffle_corr(x, y, rng_s4)
            side = L.side_shuffle_full(x, y, 6000 + si) if v == "Had" and y.sum() > 0 else None
            S4[v].append((rr, sh, side, v in r["encoded"]))
rows = []
for v, lst in S4.items():
    r_ = np.array([a[0] for a in lst])
    shm = np.vstack([a[1] for a in lst])
    ln = np.array([a[3] for a in lst])
    within = (r_ < np.nanpercentile(shm, 2.5, axis=1)) | (r_ > np.nanpercentile(shm, 97.5, axis=1))
    plo, phi = np.nanpercentile(shm, [2.5, 97.5])
    pooled = (r_ < plo) | (r_ > phi)
    if v == "Had":
        sides = [a[2] for a in lst]
        pn = np.nanpercentile(np.concatenate([s[2] for s in sides if s is not None]), [5, 95])
        pp = np.nanpercentile(np.concatenate([s[3] for s in sides if s is not None]), [5, 95])
        sym_w, sym_p = [], []
        for s in sides:
            if s is None:
                sym_w.append(False); sym_p.append(False); continue
            rn, rp, shn, shp = s
            ln5, ln95, lp5, lp95 = np.nanpercentile(shn, 5), np.nanpercentile(shn, 95), np.nanpercentile(shp, 5), np.nanpercentile(shp, 95)
            sym_w.append(bool((rn < ln5 and rp > lp95) or (rn > ln95 and rp < lp5)))
            sym_p.append(bool((rn < pn[0] and rp > pp[1]) or (rn > pn[1] and rp < pp[0])))
        within = within | np.array(sym_w)
        pooled = pooled | np.array(sym_p)
    rows.append({"var": v, "LN cells": int(ln.sum()), "within-cell": int(within.sum()), "pooled": int(pooled.sum()),
                 "LN found by within-cell": round(float(within[ln].mean()), 3) if ln.any() else np.nan,
                 "LN found by pooled": round(float(pooled[ln].mean()), 3) if ln.any() else np.nan,
                 "both > LN": bool(within.sum() > ln.sum() and pooled.sum() > ln.sum()),
                 ">= 90% by within-cell": bool(ln.any() and within[ln].mean() >= 0.9)})
t4h = pd.DataFrame(rows)
print(t4h.to_string(index=False))
print("Cells:", len(S4["Hp"]), "; with no spike (r undefined):", int(np.isnan([a[0] for a in S4["Hp"]]).sum()))

**Result: not reproduced (Cohort 1 part).** For pitch and roll, pooled shuffling finds fewer cells than the LN model (36 against 37, 35 against 43) and within-cell shuffling finds 0.811 and 0.535 of the LN cells, under 0.9. Azimuthal head velocity passes (77 and 75 cells against 10, all 10 found).

### supp5-head-model-raw-tuning-agree, Cohort 1 part

> Model-derived and raw tuning curves are highly correlated for the head variables (median 0.94 for
> pitch, 0.93 for roll, 0.90 for azimuthal head velocity).

As in the head-fixed notebook: raw curves in 20 bins of the visualizing range over the kept bins,
spikes over occupancy, smoothed with rlowess over 7 bins (Methods, "Raw tuning curves"), against the
model-derived curve at the same bin centres. Cohort 1 only, as for supp4-head.

In [ ]:
EXP5H = {"Hp": (0.94, 0.094), "Hr": (0.93, 0.093), "Had": (0.90, 0.090)}
cor5h = {v: [] for v in EXP5H}
for sid, res in RES1.items():
    P = WORK[sid][0]
    inv = np.argsort(P["order"])
    for i, r in enumerate(res):
        y = SESS[sid]["Y"][i, P["keep"]].astype(float)
        for v in EXP5H:
            if v in r["encoded"]:
                lo, hi = F.vis_range(v, P["lims"])
                c, raw = L.raw_tuning(P["vals"][v][inv], y, lo, hi)
                g, mc = r["curves"][v]
                cor5h[v].append(L.model_raw_corr(g, mc, c, raw))
rows = []
for v, (val, tol) in EXP5H.items():
    m = med(cor5h[v])
    rows.append({"var": v, "median": m[0], "IQR": m[1:3], "n": len(cor5h[v]), "not computable": int(np.isnan(cor5h[v]).sum()),
                 "paper": val, "within": bool(abs(m[0] - val) <= tol)})
t5h = pd.DataFrame(rows)
print(t5h.to_string(index=False))
print("Medians within tolerance:", int(t5h.within.sum()), "of 3 (two needed)")

**Result: reproduced (Cohort 1 part).** Median correlations pitch 0.963, roll 0.928 and azimuthal head velocity 0.816, all three within tolerance where two are needed.

## Summary

| Claim | Outcome |
|---|---|
| fig1b-head-posture-distributions | reproduced |
| fig1c-head-direction-cells | reproduced (readings 1a, 2a only) |
| fig1c-pitch-roll-cells | not reproduced (readings 1a, 2a only) |
| fig1c-pitch-roll-velocity-rare | reproduced |
| fig1d-conjunctive-head | partially reproduced |
| fig2-pitch-roll-shapes | not reproduced |
| fig2a-pitch-up-down-similar | reproduced |
| fig2b-roll-left-right-similar | reproduced |
| fig2-pitch-roll-stability | reproduced |
| supp4-head-ln-conservative | not reproduced (Cohort 1 part) |
| supp5-head-model-raw-tuning-agree | reproduced (Cohort 1 part) |
| Cohort 2 LN claims (fig1c-azimuthal-velocity-cells, the five fig2c claims, fig3ab, fig3c, fig3e, supp9-head) | blocked, too large |
| supp3-head-low-false-positives | blocked, too large |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

**How the analysis differs from the paper's.** The LN pipeline is run under one reading of open
details (1) and (2), folds in turn and out-of-range bins dropped; the claims ask for every reading,
and the others were not run, for time (31.4 min per reading under (2a); reading (2b) keeps every valid bin, about 1.44 times as many, so about 45 min). The Cohort 1 count of azimuthal head velocity cells, needed by the blocked fig1c-azimuthal-velocity-cells, is 10 (paper 12, tolerance 1.2). Every outcome on the LN model rests on
this reading. The supp4 and supp5 head parts use Cohort 1 alone. The deposit does not give the sign
convention of pitch and roll; the preference tests do not depend on it.

**What was found about the data.** The deposit's head direction is in radians, not the degrees its
description gives. Two sessions of different mice (Fuschia 052001 and Daisy 052001) have identical
body position, speed and head direction with different pitch and roll. The check of a shared clock
between spikes and behaviour was inconclusive, so the LN results on Cohort 1 rest on the deposit's
alignment, an assumption stated with each of those outcomes.

In [ ]:
print(f"Notebook time: {(time.time() - STARTED) / 60:.1f} min; peak memory {gb()} GB")